In [14]:
%reload_ext autoreload
%autoreload 2

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, DataCollatorForLanguageModeling

sentences = [
    "The weather is lovely today.",
    "It's so sunny outside!",
    "He drove to the stadium.",
]

# Load a tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

tokenized_ds = dataset.map(
        lambda examples: tokenizer(
            examples[input_config.get("dataset").get("text_column")],
            truncation=True,
        ),
        batched=True,
        num_proc=4,
        remove_columns=dataset["train"].column_names,
    )


# 1. Load a pretrained Sentence Transformer model
model = SentenceTransformer("all-MiniLM-L6-v2")

/rhome/sawale/indus_traning/mlm-fine-tuning/xenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(3, 384)
tensor([[1.0000, 0.6660, 0.1046],
        [0.6660, 1.0000, 0.1411],
        [0.1046, 0.1411, 1.0000]])


In [3]:
# The sentences to encode
sentences = [
    "The weather is lovely today.",
    "It's so sunny outside!",
    "He drove to the stadium.",
]

# 2. Calculate embeddings by calling model.encode()
embeddings = model.encode(sentences)
print(embeddings.shape)
# [3, 384]

# 3. Calculate the embedding similarities
similarities = model.similarity(embeddings, embeddings)
print(similarities)
# tensor([[1.0000, 0.6660, 0.1046],
#         [0.6660, 1.0000, 0.1411],
#         [0.1046, 0.1411, 1.0000]])

(3, 384)
tensor([[1.0000, 0.6660, 0.1046],
        [0.6660, 1.0000, 0.1411],
        [0.1046, 0.1411, 1.0000]])


In [ ]:
class KeyRanker:
    def __init__(self, sentence_transformer_model, tokenizer, tokenized_docs):
        """
        Args:
            sentence_transformer_model: Sentence Transformer model that is used to embed the documents and tokens
            tokenizer: Tokenizer that is used to tokenize the documents to tokens
            tokenized_docs: tokenized documents
        """

        def __init__(self):
            self.model = SentenceTransformer("all-MiniLM-L6-v2")
            self.tokenizer = tokenizer
            self.docs = tokenized_docs

        def generate_words(self):
            """
            Generate words from the documents
            """
            words = []
            for doc in self.docs:
                words.extend(self.tokenizer.tokenize(doc))
            return words

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")
s = "Hello people, its unbelievable."

tok.tokenize(s)

['Hello', 'Ġpeople', ',', 'Ġits', 'Ġunbelievable', '.', 'ĠPeople']

In [11]:
from transformers import AutoTokenizer, DataCollatorForLanguageModeling
import torch

# Load a tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Example texts
texts = ["Hello, how are you?", "This is a demo of DataCollatorForLanguageModeling."]

# Tokenize without `return_tensors="pt"` and keep as a list of dicts
tokenized_inputs = [tokenizer(text) for text in texts]

# add some key
for i in range(len(tokenized_inputs)):
    tokenized_inputs[i]['key'] = [1]*len(tokenized_inputs[i]['input_ids'])

print(tokenized_inputs)

# Initialize DataCollator for MLM (Masked Language Modeling)
collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=True, mlm_probability=0.15  # 15% of tokens will be masked
)

# Prepare batch data for MLM
batch = collator(tokenized_inputs)  # Now correctly passing a list of dictionaries

# Print results
print("Input IDs:\n", batch['input_ids'])
print("Labels:\n", batch['labels'])


[{'input_ids': [101, 7592, 1010, 2129, 2024, 2017, 1029, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1], 'key': [1, 1, 1, 1, 1, 1, 1, 1]}, {'input_ids': [101, 2023, 2003, 1037, 9703, 1997, 2951, 26895, 8844, 29278, 25023, 6692, 3351, 5302, 9247, 2075, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'key': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}]


ValueError: Unable to create tensor, you should probably activate truncation and/or padding with 'padding=True' 'truncation=True' to have batched tensors with the same length. Perhaps your features (`key` in this case) have excessive nesting (inputs type `list` where type `int` is expected).

In [13]:
from transformers import AutoTokenizer, DataCollatorForLanguageModeling
import torch

# Load a tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Example texts
texts = ["Hello, how are you?", "This is a demo of DataCollatorForLanguageModeling."]

# Tokenize with padding and truncation
tokenized_inputs = tokenizer(texts, padding=True, truncation=True)


print(tokenized_inputs)


# Add a new key with proper padding
for i in range(len(tokenized_inputs["input_ids"])):
    seq_len = len(tokenized_inputs["input_ids"][i])  # Get sequence length
    tokenized_inputs.setdefault("key", []).append([1] * seq_len)  # Ensure consistent shape

# Convert dictionary to a list of dicts (expected format for DataCollator)
tokenized_inputs = [{k: v[i] for k, v in tokenized_inputs.items()} for i in range(len(texts))]

# Initialize DataCollator for MLM
collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=True, mlm_probability=0.15
)

# Prepare batch data for MLM
batch = collator(tokenized_inputs)  # Now correctly passing a list of dictionaries

# Print results
print("Input IDs:\n", batch['input_ids'])
print("Labels:\n", batch['labels'])
print("Custom Key:\n", batch['key'])  # Custom key is now included properly


{'input_ids': [[101, 7592, 1010, 2129, 2024, 2017, 1029, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [101, 2023, 2003, 1037, 9703, 1997, 2951, 26895, 8844, 29278, 25023, 6692, 3351, 5302, 9247, 2075, 1012, 102]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}
Input IDs:
 tensor([[  101,  7592,  1010,  2129,  2024,  2017,   103,   102,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0],
        [  101,  2023,  2003,  1037,  9703,  1997,  2951,   103,  8844, 29278,
         25023,  6692,   103,  5302,   103,  2075,  1012,   102]])
Labels:
 tensor([[ -100,  -100,  -100,  -100,  -100,  -100,  1029,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100],
        [ -100,  -100,  -100,  -100,  -100,  -100,  -100, 26895,  -100,  -

num_proc must be <= 1. Reducing num_proc to 1 for dataset of size 1.
num_proc must be <= 1. Reducing num_proc to 1 for dataset of size 1.
Map: 100%|██████████| 1/1 [00:00<00:00, 151.63 examples/s]
num_proc must be <= 1. Reducing num_proc to 1 for dataset of size 1.
num_proc must be <= 1. Reducing num_proc to 1 for dataset of size 1.
Map: 100%|██████████| 1/1 [00:00<00:00, 217.43 examples/s]


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 8
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 1
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 1
    })
})

In [ ]:
from datasets import DatasetDict, Dataset

def preprocess_dataset(input_data, data_src, n_rows):
    # Tokenization logic (assuming tokenizer is defined)
    tokenized_data = tokenizer(input_data, truncation=True, padding="max_length")

    # Compute or assign token_importance (example: all ones)
    token_importance = [
        [1] * len(input_ids) for input_ids in tokenized_data["input_ids"]
    ]

    # Create the dataset with the new feature
    dataset = Dataset.from_dict({
        "input_ids": tokenized_data["input_ids"],
        "attention_mask": tokenized_data["attention_mask"],
        "token_importance": token_importance
    })

    return DatasetDict({"train": dataset})  # Modify as needed

preprocess_dataset()

243

# test1

In [5]:
import sys
import os
sys.path.append('/rhome/sawale/indus_traning/mlm-fine-tuning/mlm')

from preprocess_data import preprocess_dataset
import json

config_path = "../config_new_data.json"
with open(config_path, "r") as file:
    config = json.load(file)

data_src = "local"
n_rows = 10
lm_dataset, tokenizer, data_collator = preprocess_dataset(
        config.get("input"),
        data_src,
        n_rows,
    )
lm_dataset

num_proc must be <= 1. Reducing num_proc to 1 for dataset of size 1.
num_proc must be <= 1. Reducing num_proc to 1 for dataset of size 1.
num_proc must be <= 1. Reducing num_proc to 1 for dataset of size 1.
num_proc must be <= 1. Reducing num_proc to 1 for dataset of size 1.


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 8
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 1
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 1
    })
})

In [2]:
import torch
import numpy as np
from sentence_transformers import SentenceTransformer
from datasets import DatasetDict, Dataset

# Initialize the embedding model
emb_model = SentenceTransformer("all-MiniLM-L6-v2")

def compute_token_importance(examples):
    token_ids = examples["input_ids"]
    
    # Decode the full sentence
    detokenized_text = tokenizer.decode(token_ids, skip_special_tokens=True)
    
    # Decode each individual token
    tokens_decoded = tokenizer.convert_ids_to_tokens(token_ids)

    # Compute embeddings
    sentence_embedding = emb_model.encode(detokenized_text, convert_to_tensor=True)
    token_embeddings = torch.stack([
        emb_model.encode(token, convert_to_tensor=True) for token in tokens_decoded
    ])

    # Compute cosine similarity
    cosine_similarities = torch.nn.functional.cosine_similarity(sentence_embedding, token_embeddings, dim=1)
    
    # Convert to list
    return {"token_imp": cosine_similarities.tolist()}

# Apply function to each dataset split
lm_dataset = lm_dataset.map(compute_token_importance)


In [3]:
len(lm_dataset["train"][0]["input_ids"]), len(lm_dataset["train"][0]["token_imp"])

(166, 166)

In [4]:
lm_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'token_imp'],
        num_rows: 8
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'token_imp'],
        num_rows: 1
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'token_imp'],
        num_rows: 1
    })
})